In [1]:
# libraries and file paths
import pandas as pd
import pyodbc
from pathlib import Path
from getpass import getpass
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text

project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

processed_folder = project_folder / "data" / "processed"

files = [
    "cleaned_users.csv",
    "cleaned_channels.csv",
    "cleaned_videos.csv",
    "cleaned_sessions.csv",
    "cleaned_events.csv",
    "cleaned_experiment_assignments.csv"
]

missing_files = [
    file for file in files
    if not (processed_folder / file).exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing files: " + ", ".join(missing_files)
    )

print("All cleaned files were found.")

All cleaned files were found.


In [2]:
# Azure SQL connection details
driver = "ODBC Driver 18 for SQL Server"

if driver not in pyodbc.drivers():
    print("Install ODBC Driver 18 for SQL Server first.")
    print("Available drivers:", pyodbc.drivers())
    raise SystemExit

server = input(
    "Server name, example tcp:yourserver.database.windows.net,1433: "
).strip()

database = input("Database name: ").strip()
username = input("Username: ").strip()
password = getpass("Password: ")

connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect="
    + quote_plus(connection_string),
    fast_executemany=True
)

print("Connection details are ready.")

Connection details are ready.


In [3]:
# testing the connection
with engine.connect() as connection:
    database_name = connection.execute(
        text("select db_name()")
    ).scalar()

print("Connected database:", database_name)

Connected database: google_product_analytics_db


In [ ]:
# removing old staging data before loading (I failed last time loading conmplete data, want to upload in chunks now)
tables = [
    "experiment_assignments",
    "events",
    "sessions",
    "videos",
    "channels",
    "users"
]

with engine.begin() as connection:
    for table in tables:
        connection.execute(
            text(f"truncate table staging.{table}")
        )

print("Staging tables are empty.")

Staging tables are empty.


In [ ]:
# one function for loading all CSV files
def load_file(file_name, table_name, date_columns=None):
    file_path = processed_folder / file_name

    data = pd.read_csv(
        file_path,
        parse_dates=date_columns,
        low_memory=False
    )

    data.to_sql(
        table_name,
        con=engine,
        schema="staging",
        if_exists="append",
        index=False,
        chunksize=1000
    )

    print(table_name, "rows loaded:", len(data))
    return len(data)

In [6]:
# loading the smaller files first
csv_rows = {}

csv_rows["users"] = load_file(
    "cleaned_users.csv",
    "users",
    ["signup_date"]
)

csv_rows["channels"] = load_file(
    "cleaned_channels.csv",
    "channels",
    ["channel_created_date"]
)

csv_rows["videos"] = load_file(
    "cleaned_videos.csv",
    "videos",
    ["upload_date"]
)

csv_rows["sessions"] = load_file(
    "cleaned_sessions.csv",
    "sessions",
    [
        "session_start_timestamp",
        "session_end_timestamp"
    ]
)

users rows loaded: 25000
channels rows loaded: 2000
videos rows loaded: 10000
sessions rows loaded: 216464


In [7]:
# loading the events file in smaller parts (was taking a lot of time at first, now uploading in chunks)
events_file = processed_folder / "cleaned_events.csv"
events_loaded = 0

event_chunks = pd.read_csv(
    events_file,
    chunksize=50000,
    parse_dates=["event_timestamp", "event_date"],
    low_memory=False
)

for chunk_number, events in enumerate(event_chunks, start=1):
    events.to_sql(
        "events",
        con=engine,
        schema="staging",
        if_exists="append",
        index=False,
        chunksize=1000
    )

    events_loaded += len(events)

    print(
        "Chunk",
        chunk_number,
        "- total rows loaded:",
        f"{events_loaded:,}"
    )

csv_rows["events"] = events_loaded

Chunk 1 - total rows loaded: 50,000
Chunk 2 - total rows loaded: 100,000
Chunk 3 - total rows loaded: 150,000
Chunk 4 - total rows loaded: 200,000
Chunk 5 - total rows loaded: 250,000
Chunk 6 - total rows loaded: 300,000
Chunk 7 - total rows loaded: 350,000
Chunk 8 - total rows loaded: 400,000
Chunk 9 - total rows loaded: 450,000
Chunk 10 - total rows loaded: 500,000
Chunk 11 - total rows loaded: 550,000
Chunk 12 - total rows loaded: 600,000
Chunk 13 - total rows loaded: 650,000
Chunk 14 - total rows loaded: 700,000
Chunk 15 - total rows loaded: 750,000
Chunk 16 - total rows loaded: 800,000
Chunk 17 - total rows loaded: 850,000
Chunk 18 - total rows loaded: 900,000
Chunk 19 - total rows loaded: 950,000
Chunk 20 - total rows loaded: 1,000,000
Chunk 21 - total rows loaded: 1,050,000
Chunk 22 - total rows loaded: 1,100,000
Chunk 23 - total rows loaded: 1,150,000
Chunk 24 - total rows loaded: 1,200,000
Chunk 25 - total rows loaded: 1,250,000
Chunk 26 - total rows loaded: 1,300,000
Chunk 27

In [9]:
# loading experiment assignments
csv_rows["experiment_assignments"] = load_file(
    "cleaned_experiment_assignments.csv",
    "experiment_assignments",
    ["assignment_date", "first_exposure_date"]
)



experiment_assignments rows loaded: 12500


In [10]:
# comparing CSV rows with SQL rows
sql_rows = {}

table_names = [
    "users",
    "channels",
    "videos",
    "sessions",
    "events",
    "experiment_assignments"
]

with engine.connect() as connection:
    for table in table_names:
        sql_rows[table] = connection.execute(
            text(f"select count(*) from staging.{table}")
        ).scalar()

row_check = pd.DataFrame({
    "table": table_names,
    "csv_rows": [csv_rows[table] for table in table_names],
    "sql_rows": [sql_rows[table] for table in table_names]
})

row_check["rows_match"] = (
    row_check["csv_rows"] == row_check["sql_rows"]
)

display(row_check)

,table,csv_rows,sql_rows,rows_match
0,users,25000,25000,True
1,channels,2000,2000,True
2,videos,10000,10000,True
3,sessions,216464,216464,True
4,events,1609012,1609012,True
5,experiment_assignments,12500,12500,True
